In [ ]:
# IMPORT THE NECESSARY LIBRARIES / MODELS
import pandas as pd
import matplotlib.pyplot as plt 
import numpy as np 
import seaborn as sns 
import streamlit as st
import joblib
import pickle 
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix , ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor


## Here are the description on the column of this specific dataset 
- Customers who left within the last month ? the column is called Churn
- Services that each customer has signed up for ? phone, multiple lines, internet, online security, online backup, device protection, tech support, and streaming TV and movies
- Customer account information ? how long they?ve been a customer, contract, payment method, paperless billing, monthly charges, and total charges
- Demographic info about customers ? gender, age range, and if they have partners and dependents

In [ ]:
#LOADING THE DATASET 
cb_dataset = pd.read_csv("telco_data.csv")
cb_dataset.head(10)

In [ ]:
#Check info for dataset 
cb_dataset.info()



- there are no null values in this dataset 

## Question 1: What is the overall churn rate?
 What is the overall churn rate?

In [ ]:

# Set the style for the plots
sns.set(style="whitegrid")

# Create a count plot for the 'Churn' variable
plt.figure(figsize=(6, 4))
sns.countplot(x='Churn', data=cb_dataset)
plt.title('Overall Customer Churn Rate')
plt.xlabel('Churn')
plt.ylabel('Number of Customers')
plt.show()

# Calculate the churn percentage
churn_rate = cb_dataset['Churn'].value_counts(normalize=True) * 100
print(f"Churn Rate:\n{churn_rate}")



- What the Chart Shows: The first bar chart gives us a baseline. It shows that a significant majority of customers in this dataset did not churn
 

# Question 2 ; Does internet service type influence the churn rate 

In [ ]:
# Does internet service type influence the churn rate 
# Create a count plot for churn by internet service type
plt.figure(figsize=(8, 5))
sns.countplot(x='InternetService', hue='Churn', data=cb_dataset)
plt.title('Churn by Internet Service Type')
plt.xlabel('Internet Service Type')
plt.ylabel('Number of Customers')
plt.legend(title='Churn')
plt.show()

- What the Chart Shows: 

Customers with Fiber optic internet service have a much higher churn rate compared to those with DSL. Customers with no internet service have a very low churn rate.
- Why This Might Be
 
Higher Cost & Expectations: Fiber optic service is typically more expensive. Customers paying a premium may have higher expectations for speed and reliability. Any service disruption or issue could lead to greater frustration and a quicker decision to churn. The "Monthly Charges" chart supports this, as we'll see next.


## Question 3 ; Is there a difference in churn based on payment method?

In [ ]:
# Create a count plot for churn by payment method
plt.figure(figsize=(10, 6))
sns.countplot(y='PaymentMethod', hue='Churn', data=cb_dataset)
plt.title('Churn by Payment Method')
plt.xlabel('Number of Customers')
plt.ylabel('Payment Method')
plt.legend(title='Churn')
plt.show()

- What the Chart Shows:

 Customers who pay by Electronic check have a significantly higher churn rate than all other payment methods. Customers with automatic payment methods (Bank transfer or Credit card) have the lowest churn rates.
- Why This Might Be:

Manual Effort: Paying by electronic check is a manual process. Each month, the customer has to actively decide to make the payment. This monthly action is a recurring decision point where they can re-evaluate the service and decide to stop.

#  Question 4 : How does tenure relate to customer churn?

In [ ]:
# Create a box plot to see the distribution of tenure for churned and non-churned customers
plt.figure(figsize=(8, 5))
sns.boxplot(x='Churn', y='tenure', data=cb_dataset)
plt.title('Customer Tenure vs. Churn')
plt.xlabel('Churn')
plt.ylabel('Tenure (Months)')
plt.show()

- What the Chart Shows: 

The box plot clearly illustrates that customers who churn (Yes) tend to have a much shorter tenure. The median tenure for churners is low (around 10 months), while the median for non-churners is much higher (around 38 months).
- Why This Might Be:

Customer Loyalty: Loyalty builds over time. Long-term customers have likely overcome any initial issues and are satisfied with the service. They are less likely to be actively shopping for alternatives.
Early Dissatisfaction: New customers are in an evaluation period. If the service doesn't meet their expectations in the first few months (due to installation issues, billing problems, or service quality), they are highly likely to leave.

- Key Insights from Visualizations: The visualizations highlight that customers with month-to-month contracts, Fiber optic internet service, and those who use electronic checks for payment are more likely to churn. Additionally, customers with shorter tenures and higher monthly charges show a greater tendency to leave

## CREATING THE MODEL 

- data preprocessing 


In [ ]:
# Finding out whether the columns i have are necessary 
cb_dataset.columns

In [ ]:
# Finding pout the data types the columns have
cb_dataset.dtypes 


In [ ]:
# Dropping the useless columns 
cb_dataset.drop(["customerID" , "Contract"] ,axis=1 , inplace =True)
cb_dataset 

In [ ]:
#Perform encoding for the features that require the necesary numbers
# 
"""first we call the label encoder function"""
le = LabelEncoder()

"""Step 2 ; Get the feature you need to transform and actually transform it"""
cb_dataset["Dependents"] = le.fit_transform(cb_dataset["Dependents"])
cb_dataset["gender"] = le.fit_transform(cb_dataset["gender"]) #Female = 0 , Male = 1 
cb_dataset["InternetService"] = le.fit_transform(cb_dataset["InternetService"])
cb_dataset["MultipleLines"] = le.fit_transform(cb_dataset["MultipleLines"])
cb_dataset["PaymentMethod"] = le.fit_transform(cb_dataset["PaymentMethod"])
cb_dataset["TotalCharges"] = le.fit_transform(cb_dataset["TotalCharges"])
cb_dataset["MonthlyCharges"] = le.fit_transform(cb_dataset["MonthlyCharges"])
cb_dataset

In [ ]:
# Converting multiple binary columns into numbers 

#first we start by identifying the binary columns"""
binary_columns = ["Partner" , "PhoneService" , "OnlineSecurity" , "DeviceProtection" , "OnlineBackup" ,"TechSupport" , "StreamingTV" , "StreamingMovies" , "PaperlessBilling" ,"Churn"]

#The next step is we map Yes to 1  and No to 0 in each column"""
for col in binary_columns:
    cb_dataset[col] = cb_dataset[col].map({"Yes": 1 , "No": 0})

# Lets print the results and see 
cb_dataset

In [ ]:
# Check for null values 
cb_dataset.isnull().sum()

In [ ]:
# Eliminate the null vales 
cb_dataset.dropna(inplace=True)

In [ ]:
cb_dataset

In [ ]:
# Now we define what X and y are 
""" X should represent the varibales we use to predicted the value y
 y is now the value of the prediction  """

#First we define the X 
X = cb_dataset.drop(["Churn"] , axis =1).values
y = cb_dataset["Churn"]

In [ ]:
# Print X 
X

In [ ]:
cb_dataset.info 


In [ ]:
#Print y 
y 

In [ ]:
# CREATING THE MODEL . THIS IS A CLASSIFICATION MODEL 
""" First we split the data into training and testing data"""
X_train , X_test , y_train , y_test = train_test_split(X ,y , test_size=0.2 , random_state=42 , stratify=y)

#now we train our model using our training dataset 
"""but first we define our model"""
log_reg = LogisticRegression()

#now we fit the model 
log_reg.fit(X_train , y_train)

#now we make the predictions 
y_pred = log_reg.predict(X_test)

#print the predictions 
print(y_pred)


In [ ]:
"""import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler

# Let?s say your data is in a DataFrame called X (only numeric features)
# If your data includes encoded categorical variables, that?s okay

# OPTIONAL: Standardize data (improves numerical stability)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)  # or your full data

# Convert back to DataFrame (helps track feature names)
X_scaled_df = pd.DataFrame(X_scaled, columns=X_train.columns)

# Step 3: Calculate VIF for each feature
vif_data = pd.DataFrame()
vif_data['Feature'] = X_scaled_df.columns
vif_data['VIF'] = [variance_inflation_factor(X_scaled_df.values, i) for i in range(X_scaled_df.shape[1])]

# Show the results
print(vif_data.sort_values(by='VIF', ascending=False))"""


In [ ]:
# Print the accuracy of the prediction
accuracy = accuracy_score(y_test , y_pred)
print(f"Accuracy of the model :{accuracy:.2f}")

In [ ]:
# Printing the accuracy of the model 
print(" ----- Classification Report -----")
print ("Accuracy : " ,accuracy_score(y_test , y_pred))
print(classification_report(y_test, y_pred, target_names=["Not Churn", "Churn"]))


In [ ]:
# Confusion Matrix of the the Logistic Regression
# Visualization of the models prediction 
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Not Churn", "Churn"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# now we test how the model will perform using other different kind of classification models 
""" Now we import the necessary librariees and model that we can use to test my logistical model"""
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

#Now we initialize the models 
dtc = DecisionTreeClassifier(random_state=42)
rfc = RandomForestClassifier(n_estimators =100 , random_state=42)

#Now we train all the newly introduced models 
dtc.fit(X_train,y_train)
rfc.fit(X_train , y_train)

#Now we predict using all the models 
dtc_pred = dtc.predict(X_test)
rfc_pred = rfc.predict(X_test)

# We the evaluate each model 
""" first lets test the decsision tree evaluation """
print(" ----- Decision Tree ------ ")
print("Accuracy:" , accuracy_score(y_test , dtc_pred))
print(classification_report(y_test , dtc_pred , target_names= ["Not Churn" , "Churn"] ))

# Second Value to evaluate 
"""We evaluate the random forest model"""
print("---- Random Forest ------")
print("Accuracy :" , accuracy_score(y_test , rfc_pred))
print(classification_report(y_test , rfc_pred, target_names= ["Not Churn" , "Churn"] ))



In [ ]:
# Now lests compare the results and see 
results = {
    'Logistic Regression': accuracy_score(y_test, y_pred),
    'Decision Tree': accuracy_score(y_test, dtc_pred),
    'Random Forest': accuracy_score(y_test, rfc_pred)
}

print("-------- Model Accuracy Comparison ----- ")
for model, score in results.items():
    print(f"{model}: {score:.2f}")


In [ ]:
# Now we carrying out voting 

from sklearn.ensemble import VotingClassifier
voting_clf = VotingClassifier(
    estimators=[
        ('lr', log_reg),
        ('dt', dtc),
        ('rf', rfc)
    ],
    voting='soft'  # Change to 'soft' if using predict_proba
)

#Train the model 
voting_clf.fit(X_train , y_train)

# Evaluation of the model
v_pred = voting_clf.predict(X_test)

print("----- Voting Classifier -----")
print("Accuracy:", accuracy_score(y_test, v_pred))
print(classification_report(y_test, v_pred ,target_names= ["No Churn" , "Churn"]))



In [ ]:
# Visualizing the confusion Matrix after voting 
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Predict with voting classifier
y_pred = voting_clf.predict(X_test)

# Create confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=voting_clf.classes_)
disp.plot(cmap="Blues")


In [ ]:
# VISUALIZATION TWO FOR SHOWING THE ACCURACY OF THE MODEL 
# Create DataFrame for predictions
pred_df = pd.DataFrame({"Actual": y_test, "Predicted": y_pred})

# Count plot
plt.figure(figsize=(6,4))
sns.countplot(x="Actual", hue="Predicted", data=pred_df)
plt.title("Actual vs Predicted Churn")
plt.xlabel("Actual Label")
plt.ylabel("Count")
plt.legend(title="Predicted")
plt.show()


In [ ]:
from sklearn.metrics import classification_report
print("?? Classification Report:\n")
print(classification_report(y_test, y_pred, target_names=["Not Churn", "Churn"]))


- This is a confusion matrix ? a powerful tool for evaluating classification models. It shows how well your model is performing in terms of predicting churn vs. not churn.

Let's break it down in detail based on what your image shows:

|                       | **Predicted: Not Churn** | **Predicted: Churn**     |
| --------------------- | ------------------------ | ------------------------ |
| **Actual: Not Churn** | **649 (True Negative)**  | **104 (False Positive)** |
| **Actual: Churn**     | **173 (False Negative)** | **178 (True Positive)**  |



## DEFINITIONS
- True Positives (TP) = 178 ? Correctly predicted churn customers.

- True Negatives (TN) = 649 ? Correctly predicted non-churn customers.

- False Positives (FP) = 104 ? Predicted churn, but customer actually didn?t churn.

- False Negatives (FN) = 173 ? Predicted not churn, but customer actually churned.



In [ ]:
print("Expected number of features:", voting_clf.estimators_[0].n_features_in_)


In [ ]:
# Sample unseen encoded input
unseen_input = np.array([
    0, 0, 1, 0, 1, 0, 1, 0, 
    0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 
    1, 2, 142, 2505
])  # ? 18 features

unseen_input = unseen_input.reshape(1, -1)

# Predict
pred = voting_clf.predict(unseen_input)
print("Predicted class:", pred[0])


# Save the model so that we can create the interactive model


In [ ]:
# # Save the model to a file
# joblib.dump(voting_clf, "churn_model.pkl")


# # Load your model
# model = joblib.load("churn_model.pkl")

# # Title
# st.title("Customer Churn Prediction App")

# st.write("Enter customer details to predict if they are likely to churn:")

# # Input fields
# gender = st.selectbox("Gender", ["Male", "Female"])
# senior = st.selectbox("Senior Citizen?", [0, 1])
# partner = st.selectbox("Has a Partner?", ["Yes", "No"])
# dependents = st.selectbox("Has Dependents?", ["Yes", "No"])
# tenure = st.slider("Tenure (in months)", 0, 72)
# phone = st.selectbox("Phone Service", ["Yes", "No"])
# multi_lines = st.selectbox("Multiple Lines", ["Yes", "No", "No phone service"])
# internet = st.selectbox("Internet Service", ["DSL", "Fiber optic", "No"])
# online_sec = st.selectbox("Online Security", ["Yes", "No", "No internet service"])
# tech_support = st.selectbox("Tech Support", ["Yes", "No", "No internet service"])
# contract = st.selectbox("Contract Type", ["Month-to-month", "One year", "Two year"])
# paperless = st.selectbox("Paperless Billing", ["Yes", "No"])
# payment = st.selectbox("Payment Method", ["Electronic check", "Mailed check", "Bank transfer", "Credit card"])
# monthly_charges = st.number_input("Monthly Charges", min_value=0.0, step=0.1)
# total_charges = st.number_input("Total Charges", min_value=0.0, step=0.1)

# # Save the model to a file
# joblib.dump(voting_clf, "churn_model.pkl")


# # Load your model
# model = joblib.load("churn_model.pkl")



# # Assuming you already defined these somewhere above in the app:
# # gender, senior, partner, dependents, tenure, phone, multi_lines, internet, 
# # online_sec, tech_support, contract, paperless, payment, monthly_charges, total_charges

# # Prediction Button
# if st.button("Predict Churn"):
#     # Manual encoding (replace with your actual pipeline if needed)
#     encoded_input = np.array([
#         1 if gender == "Male" else 0,
#         senior,
#         1 if partner == "Yes" else 0,
#         1 if dependents == "Yes" else 0,
#         tenure,
#         1 if phone == "Yes" else 0,
#         1 if multi_lines == "Yes" else 0,
#         0 if internet == "No" else (1 if internet == "DSL" else 2),
#         1 if online_sec == "Yes" else 0,
#         1 if tech_support == "Yes" else 0,
#         0 if contract == "Month-to-month" else (1 if contract == "One year" else 2),
#         1 if paperless == "Yes" else 0,
#         0 if payment == "Electronic check" else (1 if payment == "Mailed check" else 2),
#         monthly_charges,
#         total_charges
#     ]).reshape(1, -1)

#     # Predict
#     prediction = model.predict(encoded_input)[0]
#     result = "Churn" if prediction == 1 else "Not Churn"
#     st.success(f"Prediction: The customer is likely to **{result}**.")





In [ ]:
pickle.dump(voting_clf , open('model.pkl' , 'wb'))